In [0]:
display(dbutils.fs.ls('dbfs:/databricks-datasets/structured-streaming/events'))

In [0]:
df=spark.read.format("json").load("dbfs:/databricks-datasets/structured-streaming/events/file-0.json").display()

In [0]:
df = spark.read.format("json").load("dbfs:/databricks-datasets/structured-streaming/events/")
df.count()

### From databricks-datasets/structured-streaming/events copy to target folder include checkpoint folder as well


In [0]:
target_folder="abfss://devcontainer@discoveryprodfix.dfs.core.windows.net/target_streaming"

checkpoint_folder="abfss://devcontainer@discoveryprodfix.dfs.core.windows.net/checkpoint_streaming"


In [0]:
from pyspark.sql.types import *
custom_schema = StructType([
        StructField("action",StringType(),True),
        StructField("time",LongType(),True),
    ])

In [0]:
df = spark.readStream\
      .format("json")\
      .option("maxFilesPerTrigger", 1)\
      .schema(custom_schema)\
      .option("path", "dbfs:/databricks-datasets/structured-streaming/events/")\
      .load()

In [0]:
df.writeStream\
    .format("delta")\
    .option("checkpointLocation",checkpoint_folder)\
    .option("path",target_folder)\
    .start()

In [0]:
%sql
select count(*) from delta.`abfss://devcontainer@discoveryprodfix.dfs.core.windows.net/target_streaming`

### add new column and write to folder

In [0]:
from pyspark.sql.functions import *

In [0]:

df = spark.readStream\
      .format("json")\
      .option("maxFilesPerTrigger", 1)\
      .schema(custom_schema)\
      .option("path", "dbfs:/databricks-datasets/structured-streaming/events/")\
      .load()



In [0]:
df = df.withColumn("filename", col("_metadata.file_path"))


In [0]:
df.writeStream\
    .format("delta")\
    .option("checkpointLocation",checkpoint_folder)\
    .option("path",target_folder)\
    .start()

In [0]:
%sql
select * from delta.`abfss://devcontainer@discoveryprodfix.dfs.core.windows.net/target_streaming`

In [0]:
%sql
select count(*) from delta.`abfss://devcontainer@discoveryprodfix.dfs.core.windows.net/target_streaming`